- Lecturer name: Mr.Quách Đình Hoàng 

Group member: 
- Tran Dinh Khuong - 23110035 
- Tran Huynh Xuan Thanh - 23110060

---

# Milestone 

<p align="center">
Đề tài: Phân tích cơ hội thu nhập dựa trên dữ liệu điều tra dân số năm 1994 bằng các phương pháp học máy công bằng
</p>

---


## 1.1. Thông tin về project

### • Bài toán nhóm muốn giải quyết

Bài toán đặt ra là **phân loại thu nhập cá nhân** (≤50K hoặc >50K) dựa trên các đặc trưng nhân khẩu học và nghề nghiệp từ bộ dữ liệu *Adult Census (1994)*.  

Bên cạnh mục tiêu dự đoán, nghiên cứu tập trung vào việc:
- Phân tích các yếu tố ảnh hưởng đến thu nhập  
- Đánh giá **tính công bằng (fairness)**  
- Đảm bảo **khả năng diễn giải (explainability)** của mô hình  

---

### • Tập dữ liệu, Input/Output của bài toán

- **Dữ liệu**: Adult Census (1994), gồm **48.842 quan sát** và **15 biến** (14 features, 1 target)

#### **Input (X):**
Các biến đặc trưng bao gồm:
- **Numerical**: tuổi, số năm học (*education-num*), *capital gain/loss*,…  
- **Categorical**: nghề nghiệp, trình độ học vấn, giới tính, sắc tộc,…  

#### **Output (Y):**
Biến nhị phân **income**:
- `1`: thu nhập > 50K  
- `0`: thu nhập ≤ 50K  

---

### • Cách đánh giá kết quả

Do dữ liệu **mất cân bằng lớp**, nhóm sử dụng các thước đo phù hợp hơn Accuracy:

- **F1-Score**: cân bằng giữa Precision và Recall  
- **AUC-ROC**: đánh giá khả năng phân biệt hai lớp  
- **MCC (Matthews Correlation Coefficient)**: phản ánh chất lượng phân loại tổng thể  

#### **Ngoài ra:**

- Sử dụng **Stratified K-Fold Cross Validation** để đảm bảo tính ổn định  
- Đánh giá **tính công bằng (fairness)** qua các chỉ số:
  - DPD (Demographic Parity Difference)  
  - DPR (Demographic Parity Ratio)  
  - EOD (Equalized Odds Difference)  

- Áp dụng các phương pháp **XAI**:
  - **SHAP**
  - **LIME**
  - **DiCE**

→ nhằm giải thích và phân tích hành vi của mô hình

---

## 1.2. Những kết quả mà nhóm đã đạt được. 

### 1.2.1 EDA 
<div align="center">
  <table>
    <tr>
      <td align="center">
        <img src="image/missing_values.png"/><br>
        <i>Figure 1: Missing Values</i>
      </td>
      <td align="center">
        <img src="image/class_distribution.png"/><br>
        <i>Figure 2: Class Distribution</i>
      </td>
    </tr>
  </table>
</div>

- Các quan sát chứa giá trị thiếu tập trung chủ yếu ở 3 biến: *workclass*, *occupation* và *native-country*, chiếm khoảng **7%** tổng số dữ liệu. Nhóm quyết định **loại bỏ các dòng này**, do tỷ lệ không lớn và việc loại bỏ không làm ảnh hưởng đáng kể đến thông tin tổng thể của tập dữ liệu.

- Dữ liệu có hiện tượng **mất cân bằng lớp (class imbalance)**. Cụ thể, nhóm thu nhập **≤50K chiếm khoảng 76,1%**, trong khi nhóm **>50K chỉ chiếm 23,9%**. Sự mất cân bằng này có thể khiến mô hình **thiên lệch về lớp đa số**, làm giảm khả năng dự đoán chính xác đối với lớp thiểu số.

- Để khắc phục, nhóm áp dụng kỹ thuật **SMOTE (Synthetic Minority Over-sampling Technique)** nhằm cân bằng lại phân phối dữ liệu. Đồng thời, tiến hành **so sánh hiệu suất mô hình trước và sau khi áp dụng SMOTE** để đánh giá mức độ cải thiện.

<div align="center">
  <img src="image/numerical_histograms.png" />
  <p><i>Figure 3 : numerical histograms </i></p>
</div>

- Các đặc trưng như *capital-gain* và *capital-loss* có phân phối **lệch phải (right-skewed)**. Nhóm áp dụng phương pháp **Yeo-Johnson transformation** nhằm đưa phân phối dữ liệu về gần phân phối chuẩn hơn, từ đó giúp mô hình học hiệu quả hơn.

- Các đặc trưng trong tập dữ liệu có **thang đo khác nhau đáng kể**. Để đảm bảo mô hình không bị ảnh hưởng bởi độ lớn giá trị của các biến, nhóm sử dụng **StandardScaler** để chuẩn hóa dữ liệu (đưa về cùng thang đo với mean = 0 và variance = 1).

- Biến *fnlwgt* được loại bỏ khỏi tập dữ liệu vì đây là **trọng số mẫu (sampling weight)** do cơ quan điều tra dân số cung cấp, không phản ánh trực tiếp đặc điểm của từng cá nhân và không mang nhiều ý nghĩa trong bài toán dự đoán thu nhập.


<div align="center">
  <img src="image/categorical_proportions.png" width="1400" height="800"/>
  <p><i>Figure 4: Categorical proportions</i></p>
</div>


-  Biểu đồ phân phối các biến phân loại cho thấy sự khác biệt rõ rệt về cấu trúc dữ liệu giữa các đặc trưng. Một số biến như workclass và race có sự mất cân bằng mạnh, khi một nhóm chiếm tỷ lệ áp đảo so với các nhóm còn lại. Trong khi đó, các biến như occupation và native-country có nhiều category với tần suất thấp, tiềm ẩn nguy cơ gây nhiễu và làm giảm khả năng học của mô hình.

-   Ngoài ra, một số biến thể hiện ý nghĩa thứ bậc ngầm (ví dụ liên quan đến trình độ hoặc điều kiện kinh tế), trong khi các biến khác lại hoàn toàn mang tính phân loại danh nghĩa (nominal) và không có quan hệ thứ tự.

-   Từ những đặc điểm này, có thể thấy rằng việc sử dụng một phương pháp encoding duy nhất cho tất cả các biến là không phù hợp. Thay vào đó, cần lựa chọn các phương pháp xử lý khác nhau tùy theo:

+ Mức độ mất cân bằng của biến
+ Số lượng và tần suất các category
+ Bản chất ngữ nghĩa (có thứ tự hay không)

Những quan sát này là cơ sở để đề xuất các kỹ thuật encoding phù hợp ở phần tiếp theo, nhằm vừa giảm nhiễu, vừa giữ được thông tin quan trọng phục vụ cho việc dự đoán và phân tích mô hình.

<div align="center">
  <img src="image/education_vs_education_num.png" />
</div>

- Biểu đồ phân tán giữa education và education-num cho thấy các điểm dữ liệu nằm trên một đường thẳng hoàn hảo, thể hiện mối quan hệ ánh xạ 1-1 giữa hai biến. Điều này chứng tỏ hai đặc trưng này mang cùng một thông tin dưới hai dạng biểu diễn khác nhau (dạng chữ và dạng số).

- Bên cạnh đó, biến education có nhiều category. Nếu giữ lại, việc mã hóa sẽ làm tăng số chiều dữ liệu một cách không cần thiết, trong khi không bổ sung thêm thông tin mới cho mô hình.

- Ngược lại, education-num đã thể hiện rõ tính thứ bậc tự nhiên của trình độ học vấn (giá trị càng lớn tương ứng với mức học vấn càng cao), phù hợp hơn cho các mô hình học máy.

- Từ các quan sát này, nhóm quyết định loại bỏ biến education và giữ lại education-num làm đặc trưng đại diện, nhằm giảm dư thừa thông tin và tối ưu hiệu quả mô hình.


<div align="center">
  <img src="image/cramers_v_heatmap.png"/>
</div>

- Biểu đồ tương quan cho thấy một số đặc trưng có mối liên hệ mạnh với biến mục tiêu income. Nổi bật nhất là các biến liên quan đến tình trạng gia đình (marital-status, relationship), tiếp theo là trình độ học vấn và *nghề nghiệp. Điều này cho thấy thu nhập không chỉ phụ thuộc vào yếu tố cá nhân mà còn gắn với bối cảnh xã hội và nghề nghiệp. Đây có thể là bằng chúng để phân tích các kết quả mà nhóm đạt được. 

- Bên cạnh đó, heatmap cũng chỉ ra sự tồn tại của các cặp biến có tương quan cao với nhau, đặc biệt giữa relationship, marital-status và sex. Điều này phản ánh việc các biến này đang cùng mô tả những khía cạnh gần giống nhau của cá nhân, dẫn đến nguy cơ trùng lặp thông tin.

- Từ các quan sát này, có thể thấy rằng việc giữ lại toàn bộ các biến trên có thể làm tăng nhiễu và ảnh hưởng đến độ ổn định của mô hình. Do đó,  nhóm cần tìm  hiểu nhiều phương pháp cần cân nhắc lựa chọn và loại bỏ các biến có mức tương quan cao, nhằm giảm dư thừa và cải thiện chất lượng dự đoán.

--- 
- Sau quá trình khám phá dữ liệu từ dữ liệu đầu vào thô nhóm đã thực hiện các lựa chọn đặc trưng đầu vào đồng thời tạo một dataset mới để làm input cho quá trình biến đổi các đặc trưng. (data\processed\adult_after_eda.csv)


---
### Source code 

-  Các bước phân tích khám phá dữ liệu (EDA) của nhóm được trình bày chi tiết tại. 

  `notebook/EDA_IncomeAdult.ipynb`


### 1.2.2 Feature Engineering

Dựa trên kết quả EDA, nhóm thiết kế một pipeline feature engineering theo hướng **leak-free**, trong đó toàn bộ các phép biến đổi chỉ được *fit trên tập train* và sau đó áp dụng lên tập test nhằm đảm bảo tính khách quan khi đánh giá mô hình. Pipeline gồm bốn nhóm chính: (1) biến đổi & chuẩn hoá biến số, (2) mã hoá biến phân loại, (3) tạo đặc trưng tương tác, và (4) kiểm soát data leakage.

---

#### 1. Biến đổi và chuẩn hoá biến số (Numerical Features)

Hai biến `capital-gain` và `capital-loss` có phân phối lệch phải mạnh, do đó nhóm áp dụng **Yeo-Johnson transformation** để đưa phân phối về gần chuẩn mà không yêu cầu dữ liệu dương. Tham số ước lượng:

- λ(capital-gain) = -1.37  
- λ(capital-loss) = -2.85  

Ba biến `age`, `hours-per-week`, `education-num` được chuẩn hoá bằng **RobustScaler** (median + IQR) thay vì StandardScaler do tồn tại outliers. Điều này giúp mô hình ổn định hơn trước các giá trị ngoại lai.

---

#### 2. Mã hoá biến phân loại (Categorical Encoding)

Các biến categorical được xử lý linh hoạt theo đặc điểm dữ liệu:

- **Ordinal Encoding**:  
  `native-country → country_income_group (0–3)` dựa trên phân loại thu nhập quốc gia (World Bank), giúp giảm số category và giữ thứ bậc kinh tế.

- **Grouping + CatBoost Encoding**:  
  `occupation` (14 categories) → gom thành 4 nhóm (`manual`, `high_skill`, `office_tech`, `specialised`) → encode bằng **CatBoost Encoder** để tránh leakage.

- **Leave-One-Out Encoding (LOO)**:  
  `workclass` (mất cân bằng mạnh) → encode bằng trung bình target (trừ chính sample đó) → giảm overfitting so với target encoding.

- **One-Hot Encoding**:  
  - `relationship` → 6 features  
  - `race` → 5 features  

- **Binary Encoding**:  
  - `sex → sex_binary (1 = Male, 0 = Female)`  
  - `marital-status → married_flag (1/0)`

---

#### 3. Tạo đặc trưng tương tác (Feature Interactions)

Nhóm tạo thêm **18 features mới**:

**(a) Econometric features (3 features)**  
- `human_capital = age × education-num`  
- `household_labour = hours-per-week × married_flag`  
- `net_capital = capital-gain − capital-loss`  

**(b) Fairness interactions (15 features)**  
Cross giữa:
- `education-num × race` → 5 features  
- `hours-per-week × race` → 5 features  
- `net_capital × race` → 5 features  

→ Mục tiêu: phân tích chênh lệch giữa các nhóm nhân khẩu học (fairness analysis).

---

#### 4. Tổng số đặc trưng sau xử lý

| Nhóm đặc trưng                  | Số lượng | Kỹ thuật |
|--------------------------------|----------|----------|
| Numerical (sau transform)      | 5        | Yeo-Johnson, Robust Scaling |
| Categorical đã mã hoá          | ~10      | Ordinal, LOO, CatBoost, Binary |
| One-Hot (relationship + race)  | 11       | One-Hot Encoding |
| Econometric features           | 3        | Feature interaction |
| Fairness interactions          | 15       | Cross-product với race |
| **Tổng cộng**                  | **~40**  | — |

<div align="center">
  <img src="image/Feature_engineering.png" width="1400" height="800"/>
</div>

---



#### 5. Kiểm soát Data Leakage

Toàn bộ các transformer và encoder:
- Yeo-Johnson  
- RobustScaler  
- CatBoost Encoder  
- LOO Encoder  

đều được **fit trên tập train (39,073 samples)** và **chỉ transform trên test (9,769 samples)**.

→ Đảm bảo:
- Không rò rỉ thông tin  
- Đánh giá mô hình phản ánh đúng khả năng tổng quát hoá  
<div align="center">
  <table>
    <tr>
      <td align="center">
        <img src="image/Category_engin.png"/><br>
      </td>
      <td align="center">
        <img src="image/Tranform.png"/><br>
      </td>
    </tr>
  </table>
</div>

- Feature engineering chủ yếu làm tăng số lượng đặc trưng thông qua các biến tương tác và các biến phục vụ phân tích công bằng (fairness), thay vì chỉ đơn thuần là các phương pháp mã hóa.

- ► Nhiều kỹ thuật encoding và scaling khác nhau được áp dụng, nhưng phần lớn các đặc trưng mới được tạo ra từ one-hot encoding và các biến tương tác.

---

# Source code các thực hiện về feature engineering của nhóm 
`notebook\feature_engineering_pipeline.ipynb`

---

###  1.2.3Model Building & Evaluation

    